In [0]:
dbutils.widgets.text("storage_account_name", "", "ADLS Storage Account Name")
dbutils.widgets.text("storage_account_key", "", "ADLS Storage Account Key")

In [0]:
storage_account_name = dbutils.widgets.get("storage_account_name")
storage_account_key = dbutils.widgets.get("storage_account_key")

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key,
)

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/trades/"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/trades_pyspark/"


In [0]:
landing_path = f"abfss://landing@{storage_account_name}.dfs.core.windows.net/trades/"

In [0]:
# Use wasbs:// (Blob endpoint) to avoid DFS endpoint BlobStorageEvents/SoftDelete error
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key,
)
landing_path_blob = f"wasbs://landing@{storage_account_name}.blob.core.windows.net/trades/"

# List all files at the landing path
files = dbutils.fs.ls(landing_path_blob)
print("Files at landing path:")
for f in files:
    print(f"  {f.name} | Size: {f.size} bytes")

# Read each file one by one into a Spark DataFrame
for f in files:
    file_path = f.path
    print(f"\nReading file: {f.name}")
    df = spark.read.csv(file_path, header=True, inferSchema=True)
    df.show(5)
    df.printSchema()

Files at landing path:
  trade_blotter.csv | Size: 33567 bytes

Reading file: trade_blotter.csv
+--------+----------+-----------+---------+----------+--------+-------+-------------------+
|trade_id|trade_date|security_id|trader_id|trade_type|quantity|  price|   last_modified_ts|
+--------+----------+-----------+---------+----------+--------+-------+-------------------+
|  100224|2026-05-19|          1|        6|      SELL|     500|1155.56|2026-05-19 17:43:00|
|  100137|2026-05-13|          2|        4|       BUY|    4200|2688.04|2026-05-13 16:59:00|
|  100112|2026-05-11|          5|        6|       BUY|    4100|2798.81|2026-05-11 16:06:00|
|  100496|2026-06-09|          8|        2|       BUY|    4000|2038.49|2026-06-09 12:41:00|
|  100254|2026-05-22|          5|        1|       BUY|    1900|1238.77|2026-05-22 09:55:00|
+--------+----------+-----------+---------+----------+--------+-------+-------------------+
only showing top 5 rows
root
 |-- trade_id: integer (nullable = true)
 |-- t

In [0]:
print(f"Cluster default parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Partitions after reading: {df.rdd.getNumPartitions()}")

Cluster default parallelism: 4
Partitions after reading: 1


In [0]:
bronze_path = f"wasbs://landing@{storage_account_name}.blob.core.windows.net/trades/"
df.write.mode("overwrite").format("parquet").save(bronze_path)

In [0]:
bronze_df = spark.read.parquet(bronze_path)
print(f"Partitions after reading: {bronze_df.rdd.getNumPartitions()}")  

Partitions after reading: 1


In [0]:
repartitioned = bronze_df.repartition(8, "security_id")

In [0]:
repartitioned.rdd.getNumPartitions()

8

In [0]:
bronze_path = f"wasbs://landing@{storage_account_name}.blob.core.windows.net/trades_part1/"
repartitioned.write.mode("overwrite").format("parquet").save(bronze_path)